# 护栏

为您的代理实施安全检查和内容过滤。

防护机制通过在代理执行的关键节点验证和过滤内容，帮助您构建安全合规的 AI 应用。它们可以检测敏感信息、强制执行内容策略、验证输出，并在不安全行为造成问题之前加以阻止。

常见应用场景包括：
- 防止个人身份信息泄露
- 检测和阻止提示注入攻击
- 屏蔽不当或有害内容
- 执行业务规则和合规要求
- 验证输出质量和准确性
- 您可以使用中间件来实现防护措施，在关键点拦截执行——在代理启动之前、完成之后，或者在模型和工具调用前后。


您可以使用中间件来实现防护措施，在关键点拦截执行——在代理启动之前、完成之后，或者在模型和工具调用前后。

<img src="https://mintcdn.com/langchain-5e9cc07a/RAP6mjwE5G00xYsA/oss/images/middleware_final.png?w=1100&fit=max&auto=format&n=RAP6mjwE5G00xYsA&q=85&s=ec45e1932d1279b1beee4a4b016b473f">

防护措施可以通过两种互补的方法来实现：

- 确定性护栏.  
使用基于规则的逻辑，例如正则表达式模式、关键字匹配或显式检查。这种方法速度快、结果可预测且成本效益高，但可能会遗漏一些细微的违规行为。

- 基于模型的护栏.  
使用逻辑逻辑模型或分类器来评估具有语义理解的内容。它们可以捕捉到规则无法发现的细微问题，但速度较慢且成本更高。


LangChain 既提供了内置的防护措​​施（例如，PII 检测、人机交互），也提供了一个灵活的中间件系统，可以使用这两种方法构建自定义防护措施。
​


## 内置护栏
​
### PII 检测

LangChain 提供内置中间件，用于检测和处理对话中的个人身份信息 (PII)。该中间件可以检测常见的 PII 类型，例如电子邮件、信用卡信息、IP 地址等。

PII 检测中间件对医疗保健和金融等有合规性要求的应用、需要清理日志的客户服务代理以及处理敏感用户数据的任何应用都很有帮助。

PII 中间件支持多种处理检测到的 PII 的策略：

| 战略 | 描述 | 例子 |
| :--- | :--- | :--- |
| redact | 替换为［REDACTED＿TYPE］ | ［REDACTED＿EMAIL］ |
| mask | 部分模糊不清（例如，最后 4 位数字） | ＊＊＊＊－＊＊＊＊－＊＊＊＊－1234 |
| hash | 替换为确定性哈希 | a8f5f167．．． |
| block | 检测到异常时抛出异常 | 抛出错误 |

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware


agent = create_agent(
    model="gpt-4o",
    tools=[customer_service_tool, email_tool],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

# When user provides PII, it will be handled according to the strategy
result = agent.invoke({
    "messages": [{"role": "user", "content": "My email is john.doe@example.com and card is 4532-1234-5678-9010"}]
})

内置PII类型：
- email- 电子邮件地址
- credit_card- 信用卡号码（Luhn 验证）
- ip- IP地址
- mac_addressMAC地址
- url- 网址

| 范围 | 描述 |  | 默认 |
| :--- | :--- | :--- | :--- |
| pii＿type | 要检测的个人身份信息类型（内置或自定义） |  | 必需的 |
| strategy | 如何处理检测到的个人身份信息（＂block＂，，，＂redact＂） | ＂mask＂＂hash＂ | ＂redact＂ |
| detector | 自定义检测器函数或正则表达式模式 |  | None（使用）数） |
| apply_to_input | 在模型调用之前检查用户消息 |  | True |
| apply_to_output | 模型调用后检查 AI 消息 |  | False |
| apply_to_tool_results | 执行后检查工具结果消息 |  | False |

有关 PII 检测功能的完整详细信息，请参阅中间件文档。

### 人机交互
LangChain 提供内置中间件，用于在执行敏感操作前要求人工审批。这是高风险决策最有效的保障措施之一。

人机交互中间件有助于处理诸如金融交易和转账、删除或修改生产数据、向外部各方发送通信以及任何具有重大业务影响的操作等情况。


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, send_email_tool, delete_database_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval for sensitive operations
                "send_email": True,
                "delete_database": True,
                # Auto-approve safe operations
                "search": False,
            }
        ),
    ],
    # Persist the state across interrupts
    checkpointer=InMemorySaver(),
)

# Human-in-the-loop requires a thread ID for persistence
config = {"configurable": {"thread_id": "some_id"}}

# Agent will pause and wait for approval before executing sensitive tools
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to the team"}]},
    config=config
)

result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Same thread ID to resume the paused conversation
)

## 定制护栏
对于更复杂的防护措施，您可以创建自定义中间件，使其在代理执行之前或之后运行。这样，您就可以完全控制验证逻辑、内容过滤和安全检查。

### 在代理护栏之前
使用“before_agent”钩子在每次调用开始时验证请求一次。这对于会话级检查（例如身份验证、速率限制或在任何处理开始之前阻止不当请求）非常有用。

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime

class ContentFilterMiddleware(AgentMiddleware):
    """Deterministic guardrail: Block requests containing banned keywords."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the first user message
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        # Check for banned keywords
        for keyword in self.banned_keywords:
            if keyword in content:
                # Block execution before any processing
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": "I cannot process requests containing inappropriate content. Please rephrase your request."
                    }],
                    "jump_to": "end"
                }

        return None

# Use the custom guardrail
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, calculator_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware"]
        ),
    ],
)

# This request will be blocked before any processing
result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a database?"}]
})

代理护栏之后

使用“after_agent”钩子在返回给用户之前对最终输出进行一次验证。这对于基于模型的安全检查、质量验证或对完整代理响应进行最终合规性扫描非常有用。

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.chat_models import init_chat_model
from typing import Any

class SafetyGuardrailMiddleware(AgentMiddleware):
    """Model-based guardrail: Use an LLM to evaluate response safety."""

    def __init__(self):
        super().__init__()
        self.safety_model = init_chat_model("gpt-4o-mini")

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the final AI response
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a model to evaluate safety
        safety_prompt = f"""Evaluate if this response is safe and appropriate.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response: {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "I cannot provide that response. Please rephrase your request."
                }],
                "jump_to": "end"
            }

        return None

# Use the safety guardrail
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, calculator_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I make explosives?"}]
})

组合多个护栏

您可以通过将多个防护措施添加到中间件数组中来堆叠它们。它们按顺序执行，从而允许您构建分层保护：

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware

agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit"]),

        # Layer 2: PII protection (before and after model)
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),

        # Layer 4: Model-based safety check (after agent)
        SafetyGuardrailMiddleware(),
    ],
)